# Étape 1 — Test de la synthèse vocale éwé (Meta MMS)

**Projet ɖevi ƒe agbalẽ · Hack4Children 2026**

But : décider **ce soir** si `facebook/mms-tts-ewe` est assez bon pour les messages aux familles.

Ce notebook :
1. installe et charge le modèle ;
2. vérifie que les lettres éwé (ɖ, ƒ, ɣ, ŋ, ɔ, ɛ, ʋ, accents) sont bien lues ;
3. génère quelques phrases et des prénoms, à écouter ;
4. compare deux vitesses de parole ;
5. télécharge tous les fichiers audio dans un .zip.

> Dans Colab : *Exécution > Modifier le type d'exécution > GPU T4* (facultatif, le processeur suffit).

In [2]:
# 1. Installation (torch est déjà présent dans Colab)
!pip install -q transformers scipy

In [3]:
# 2. Chargement du modèle éwé
import torch, unicodedata, os
from transformers import VitsModel, AutoTokenizer, set_seed

MODELE = "facebook/mms-tts-ewe"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODELE)
model = VitsModel.from_pretrained(MODELE).to(device).eval()
SR = model.config.sampling_rate

print("Appareil :", device, "| fréquence :", SR, "Hz")
print("Romanisation (uroman) nécessaire :", getattr(tokenizer, "is_uroman", False))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Appareil : cpu | fréquence : 16000 Hz
Romanisation (uroman) nécessaire : False


## 3. Vérifier l'alphabet

Le modèle **supprime en silence** tout caractère absent de son vocabulaire.
Si une lettre éwé manque, un son disparaît sans message d'erreur. On vérifie donc avant tout.

In [4]:
vocab = set(tokenizer.get_vocab().keys())
print("Caractères connus du modèle :")
print(" ".join(sorted(c for c in vocab if len(c) == 1)))

def caracteres_inconnus(texte, forme="NFC"):
    t = unicodedata.normalize(forme, texte.lower())
    return sorted({c for c in t if c not in vocab and not c.isspace()})

LETTRES_EWE = "ɖ ƒ ɣ ŋ ɔ ɛ ʋ é è ê à á ã ẽ õ ò ó ù ú ì í"
for forme in ("NFC", "NFD"):
    manquants = caracteres_inconnus(LETTRES_EWE, forme)
    print(f"\nForme {forme} - caractères NON reconnus :",
          [f"{c} (U+{ord(c):04X})" for c in manquants] or "aucun")

Caractères connus du modèle :
  ' - _ a b d e f g h i k l m n o p r s t u v w x y z à á ã è é ì í ò ó ù ú ĩ ŋ ũ ƒ ɔ ɖ ɛ ɣ ʋ ̀ ́ ̃ ẽ

Forme NFC - caractères NON reconnus : ['ê (U+00EA)', 'õ (U+00F5)']

Forme NFD - caractères NON reconnus : ['̂ (U+0302)']


**Résultat du 16 nov. :** toutes les lettres éwé sont reconnues. Seuls ê (NFC) et õ (NFC) posent problème : la cellule suivante les corrige.

In [5]:
# Règle retenue après la vérification de l'alphabet :
# - forme NFC (lettres accentuées composées, comme dans le vocabulaire du modèle)
# - õ n'existe pas composé -> o + tilde combinant (connu du modèle)
# - ê : le circonflexe est inconnu -> remplacé par e (ton montant-descendant perdu, cas rare)
REMPLACEMENTS = {"õ": "õ", "ê": "e"}
PONCTUATION = set(".,!?;:")   # sert à couper le texte en phrases (pauses), jamais lue par le modèle

def normaliser(texte):
    t = unicodedata.normalize("NFC", texte.lower())
    for avant, apres in REMPLACEMENTS.items():
        t = t.replace(avant, apres)
    return t

def caracteres_restants(texte):
    """Caractères qui seront supprimés en silence par le modèle."""
    return sorted({c for c in normaliser(texte) if c not in vocab and not c.isspace()})

def verifier(texte):
    """Bloque tout texte qui perdrait des sons (c, j, q, chiffres, symboles)."""
    perdus = [c for c in caracteres_restants(texte) if c not in PONCTUATION]
    if perdus:
        raise ValueError(
            f"Caractères non lus par le modèle : {perdus} dans « {texte} ». "
            "Remplacer c par k ou s, j par dz, écrire les nombres en toutes lettres (ɖeka, eve, etɔ̃...)."
        )

def texte_vu_par_le_modele(texte):
    ids = tokenizer(normaliser(texte))["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    return "".join(x for x in toks if x != tokenizer.pad_token)

print(texte_vu_par_le_modele("Ɖevi ƒe agbalẽ, Kodzo, õ, ê, ŋkeke eve."))

ɖevi ƒe agbalẽ kodzo õ e ŋkeke eve


**Ce que le modèle ne lit pas (constaté au premier essai) :**
- **pas de c** → « vaccin » devenait « vain » : écrire **vaksin** (ou le mot éwé de ton choix) ;
- **pas de chiffres** → « zi 1 » devenait « zi » : écrire **zi ɖeka**, **zi eve** ;
- **pas de j, q** → prénoms en orthographe éwé (**Kodzo**, **Kosi**) ;
- **pas de ponctuation** → la fonction `synthese` coupe désormais le texte aux signes `. , ! ? ; :` et ajoute une **pause** entre les morceaux.

`verifier()` bloque tout texte qui perdrait encore des sons : corrige le texte jusqu'à ce qu'il passe.

## 4. Phrases de test

**Jérôme : remplace les phrases éwé** par tes propres traductions (garde l'orthographe complète, avec tons si tu les écris).
Le français est là pour mémoire. Les phrases ci-dessous ne sont que des exemples de départ.

In [22]:
PHRASES = {
    "bienvenue": {
        "fr": "Bienvenue. Votre enfant est maintenant inscrit dans le carnet ɖevi ƒe agbalẽ.",
        "ewe": "Woèzɔnlo! Ŋdi na wò. Wotsɔ viò ƒe ŋkɔ ŋlɔ ɖe ɖevi ƒe agbalẽ la me fifia.",
    },
    "rappel_vaccin": {
        "fr": "Après-demain, votre enfant doit recevoir son vaccin au centre de santé. Ce vaccin est gratuit.",
        "ewe": "Le ŋkeke eve si gbɔna me la, mia vi la axɔ ye ƒe Abɔta ssi-ssi le dɔyɔƒe. Abɔta la , fe aɖeke mele eŋu o ",
       # Le ŋkeke eve si gbɔna me la, mia  ƒe vi dò la xɔ ye ƒe abɔta le dɔyɔƒe. Abɔta la , fe aɖeke mele eŋu o !
    },
    "rendez_vous_manque": {
        "fr": "Votre enfant n'est pas venu à son rendez-vous de vaccin. Appuyez sur 1 si vous viendrez, sur 2 si vous avez un empêchement.",
        "ewe": "mi mé va do mia vi la ƒe abɔta-lao le eyĩ-ĩɣi si oɖo ne la o. Zi ŋu ŋlɔ-ŋlɔ ɖeka la di, ne mi àva dɔyɔƒe. zi ŋu ŋlɔ-ŋlɔ e-ve, ne nu aɖe xe emɔ na mi.",
    },
}
# « vaksin » et « ɖeka / eve » corrigés d'après le premier essai : modifie librement le choix des mots.

for cle, p in PHRASES.items():
    if p["ewe"]:
        verifier(p["ewe"])
        print(f"{cle:20s} | OK | vu par le modèle : {texte_vu_par_le_modele(p['ewe'])}")

bienvenue            | OK | vu par le modèle : woèzɔnlo ŋdi na wò wotsɔ viò ƒe ŋkɔ ŋlɔ ɖe ɖevi ƒe agbalẽ la me fifia
rappel_vaccin        | OK | vu par le modèle : le ŋkeke eve si gbɔna me la mia vi la axɔ ye ƒe abɔta ssi-ssi le dɔyɔƒe abɔta la  fe aɖeke mele eŋu o
rendez_vous_manque   | OK | vu par le modèle : mi mé va do mia vi la ƒe abɔta-lao le eyĩ-ĩɣi si oɖo ne la o zi ŋu ŋlɔ-ŋlɔ ɖeka la di ne mi àva dɔyɔƒe zi ŋu ŋlɔ-ŋlɔ e-ve ne nu aɖe xe emɔ na mi


In [23]:
# 5. Génération et écoute (avec pauses entre les phrases)
import re
import numpy as np, scipy.io.wavfile as wavfile
from IPython.display import Audio, display

SORTIE = "/content/voix_test"
os.makedirs(SORTIE, exist_ok=True)

PAUSE_PHRASE = 0.45   # secondes après . ! ? ; :
PAUSE_VIRGULE = 0.20  # secondes après ,

def _synthese_brute(texte, vitesse, graine):
    set_seed(graine)                 # même graine = même voix à chaque fois
    model.speaking_rate = vitesse    # < 1 : plus lent, > 1 : plus rapide
    entree = tokenizer(normaliser(texte), return_tensors="pt").to(device)
    with torch.no_grad():
        return model(**entree).waveform[0].cpu().numpy()

def decouper(texte):
    """Coupe le texte aux signes de ponctuation : [(morceau, pause_en_secondes), ...]."""
    morceaux = re.split(r"([.!?;:]+|,)", texte)
    resultat = []
    for i in range(0, len(morceaux), 2):
        bout = morceaux[i].strip()
        signe = morceaux[i + 1] if i + 1 < len(morceaux) else ""
        if bout:
            pause = PAUSE_VIRGULE if signe == "," else (PAUSE_PHRASE if signe else 0.0)
            resultat.append((bout, pause))
    return resultat

def synthese(texte, vitesse=1.0, graine=42, pauses=True):
    """Signal audio (numpy) d'un texte éwé ; une pause est insérée à chaque ponctuation."""
    verifier(texte)
    if not pauses:
        return _synthese_brute(texte, vitesse, graine)
    parties = []
    for bout, pause in decouper(texte):
        parties.append(_synthese_brute(bout, vitesse, graine))
        if pause:
            parties.append(np.zeros(int(pause * SR), dtype=np.float32))
    return np.concatenate(parties)

def sauver(onde, nom):
    chemin = f"{SORTIE}/{nom}.wav"
    wavfile.write(chemin, SR, (np.clip(onde, -1, 1) * 32767).astype(np.int16))
    return chemin

for cle, p in PHRASES.items():
    if not p["ewe"]:
        print(f"[{cle}] pas encore de texte éwé - ignoré")
        continue
    onde = synthese(p["ewe"])
    sauver(onde, cle)
    print(f"[{cle}] {len(onde)/SR:.1f} s - {p['fr']}")
    display(Audio(onde, rate=SR))

[bienvenue] 8.4 s - Bienvenue. Votre enfant est maintenant inscrit dans le carnet ɖevi ƒe agbalẽ.


[rappel_vaccin] 12.8 s - Après-demain, votre enfant doit recevoir son vaccin au centre de santé. Ce vaccin est gratuit.


[rendez_vous_manque] 17.0 s - Votre enfant n'est pas venu à son rendez-vous de vaccin. Appuyez sur 1 si vous viendrez, sur 2 si vous avez un empêchement.


## 5 bis. Avec ou sans pauses

Même phrase, deux versions : écoute la différence de naturel.

In [24]:
p = PHRASES["rendez_vous_manque"]["ewe"]
for avec in (False, True):
    onde = synthese(p, pauses=avec)
    sauver(onde, f"rendez_vous_manque_{'avec' if avec else 'sans'}_pauses")
    print("Avec pauses" if avec else "Sans pauses", f"- {len(onde)/SR:.1f} s")
    display(Audio(onde, rate=SR))

Sans pauses - 12.2 s


Avec pauses - 17.0 s


## 6. Vitesse de parole

Pour des mères qui écoutent au téléphone, un débit un peu plus lent peut être plus clair. Compare 0,85 et 1,0.

In [25]:
phrase_test = next((p["ewe"] for p in PHRASES.values() if p["ewe"]), "")
for v in (0.85, 1.0):
    onde = synthese(phrase_test, vitesse=v)
    sauver(onde, f"vitesse_{v}")
    print(f"Vitesse {v}")
    display(Audio(onde, rate=SR))

Vitesse 0.85


Vitesse 1.0


## 7. Prénoms

Les prénoms seront générés à la volée dans les messages. Ils doivent être écrits en orthographe éwé (pas de c, j, q).

In [10]:
PRENOMS = ["Kɔssi", "A-ma", "Yawo", "Affi", "Kɔdzo", "Ako-ssi-wa", "Kɔmi", "Abla", "Mawuli", "Se-na"]   # orthographe éwé
for p in PRENOMS:
    if caracteres_restants(p):
        print("Attention", p, ":", caracteres_restants(p))

for prenom in PRENOMS:
    onde = synthese(prenom, vitesse=0.9)
    sauver(onde, f"prenom_{prenom}")
    print(prenom)
    display(Audio(onde, rate=SR))

Kɔssi


A-ma


Yawo


Affi


Kɔdzo


Ako-ssi-wa


Kɔmi


Abla


Mawuli


Se-na


## 8. (Facultatif) Essai rapide en kabiyè

Pour la finale. Laisse vide si personne ne peut juger la prononciation pour l'instant.

In [28]:
PHRASE_KABIYE = ""   # une phrase kabiyè à tester

if PHRASE_KABIYE:
    tok_kbp = AutoTokenizer.from_pretrained("facebook/mms-tts-kbp")
    mod_kbp = VitsModel.from_pretrained("facebook/mms-tts-kbp").to(device).eval()
    set_seed(42)
    with torch.no_grad():
        onde = mod_kbp(**tok_kbp(PHRASE_KABIYE, return_tensors="pt").to(device)).waveform[0].cpu().numpy()
    sauver(onde, "kabiye_test")
    display(Audio(onde, rate=mod_kbp.config.sampling_rate))

In [26]:
# 9. Télécharger tous les fichiers générés (.zip)
import shutil
from google.colab import files
archive = shutil.make_archive("/content/voix_test_mms_ewe", "zip", SORTIE)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Grille de décision (à remplir après écoute)

| Critère | Note /5 | Remarque |
|---|---|---|
| Compréhensible du premier coup | | |
| Tons et voyelles corrects (ɔ, ɛ, nasales) | | |
| Prénoms bien prononcés | | |
| Les pauses rendent la voix plus naturelle | | |
| Vitesse retenue (0,85 ou 1,0) | | |

**Choix retenu (16 nov.) : MMS pour tout**, texte écrit pour le modèle et pauses entre les phrases.
Si un message reste incompréhensible malgré tout : Jérôme enregistre ce message **en entier** (sans partie variable), pour ne jamais mélanger deux voix dans une même phrase.